# NASA C-MAPSS FD001 Exploratory Data Analysis

This notebook validates the first engine dataset, calculates training RUL, identifies uninformative sensors, and examines degradation patterns. FD001 is the first reference pipeline because it has one operating condition and one fault mode.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'PROJECT_ROADMAP.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.cmapss import load_training_with_rul

FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures' / 'engine'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

## Load validated training data

The project loader enforces the expected 26-column schema, positive unit/cycle identifiers, no missing values, and no duplicate unit-cycle pairs.

In [ ]:
SUBSET = 'FD001'
engine = load_training_with_rul(SUBSET)
sensor_columns = [column for column in engine.columns if column.startswith('sensor_')]
setting_columns = [column for column in engine.columns if column.startswith('operational_setting_')]
engine.head()

In [ ]:
integrity = pd.Series({
    'rows': len(engine),
    'engines': engine['unit_id'].nunique(),
    'source_columns': len(engine.columns) - 1,
    'sensor_columns': len(sensor_columns),
    'missing_values': int(engine.isna().sum().sum()),
    'duplicate_unit_cycles': int(engine.duplicated(['unit_id', 'cycle']).sum()),
    'minimum_rul': int(engine['rul'].min()),
    'maximum_rul': int(engine['rul'].max()),
}, name='value')
integrity.to_frame()

## Engine lifetime distribution

Each training trajectory reaches failure, so its maximum observed cycle is its recorded lifetime.

In [ ]:
lifetimes = engine.groupby('unit_id', as_index=False)['cycle'].max().rename(columns={'cycle': 'lifetime_cycles'})
lifetimes['lifetime_cycles'].describe().to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(lifetimes['lifetime_cycles'], bins=15, color='#0891b2', edgecolor='white')
ax.axvline(lifetimes['lifetime_cycles'].median(), color='#f59e0b', linestyle='--', label=f"Median: {lifetimes['lifetime_cycles'].median():.0f} cycles")
ax.set(title='FD001 Engine Lifetime Distribution', xlabel='Recorded lifetime (cycles)', ylabel='Number of engines')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'fd001_engine_lifetimes.png', dpi=160, bbox_inches='tight')
plt.show()

## Sensor distributions and variance

Constant and nearly constant measurements add no useful degradation information in FD001. They will be candidates for removal by a fitted preprocessing pipeline.

In [ ]:
sensor_profile = engine[sensor_columns].agg(['min', 'max', 'mean', 'std']).T
sensor_profile['unique_values'] = engine[sensor_columns].nunique()
sensor_profile['range'] = sensor_profile['max'] - sensor_profile['min']
constant_sensors = sensor_profile.index[sensor_profile['unique_values'] <= 1].tolist()
near_constant_sensors = sensor_profile.index[(sensor_profile['unique_values'] > 1) & (sensor_profile['std'] < 1e-6)].tolist()
print('Constant sensors:', constant_sensors)
print('Near-constant sensors:', near_constant_sensors)
sensor_profile

In [ ]:
variable_sensors = [column for column in sensor_columns if column not in constant_sensors + near_constant_sensors]
plot_sensors = variable_sensors[:12]
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for axis, column in zip(axes.flat, plot_sensors):
    axis.hist(engine[column], bins=35, color='#38bdf8', alpha=0.85)
    axis.set_title(column)
    axis.tick_params(labelsize=8)
fig.suptitle('FD001 Variable Sensor Distributions', fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'fd001_sensor_distributions.png', dpi=160, bbox_inches='tight')
plt.show()

## Relationship between sensors and RUL

Correlation is an exploratory linear association, not proof of causation or a final feature-selection rule.

In [ ]:
rul_correlations = (
    engine[variable_sensors + ['rul']]
    .corr(numeric_only=True)['rul']
    .drop('rul')
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
rul_correlations.to_frame('pearson_correlation_with_rul')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ordered = rul_correlations.sort_values()
colors = ['#ef4444' if value < 0 else '#0ea5e9' for value in ordered]
ax.barh(ordered.index, ordered.values, color=colors)
ax.axvline(0, color='#334155', linewidth=1)
ax.set(title='FD001 Sensor Correlation with RUL', xlabel='Pearson correlation', ylabel='Sensor')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'fd001_sensor_rul_correlations.png', dpi=160, bbox_inches='tight')
plt.show()

## Degradation trends on a common lifecycle scale

Engines have different lifetimes. The following chart aligns them using lifecycle progress from 0 (first observation) to 1 (failure), then standardizes each selected sensor for visual comparison.

In [ ]:
top_sensors = rul_correlations.head(4).index.tolist()
lifecycle = engine[['unit_id', 'cycle', *top_sensors]].copy()
lifecycle['max_cycle'] = lifecycle.groupby('unit_id')['cycle'].transform('max')
lifecycle['life_fraction'] = lifecycle['cycle'] / lifecycle['max_cycle']
lifecycle['life_bin'] = pd.cut(lifecycle['life_fraction'], bins=np.linspace(0, 1, 21), include_lowest=True, labels=False)
means = lifecycle.groupby('life_bin', observed=True)[top_sensors].mean()
standardized = (means - means.mean()) / means.std(ddof=0)
standardized.index = (standardized.index + 0.5) / 20

fig, ax = plt.subplots(figsize=(10, 5))
for column in top_sensors:
    ax.plot(standardized.index, standardized[column], marker='o', markersize=3, label=column)
ax.set(title='Mean Sensor Degradation Across Normalized Engine Life', xlabel='Lifecycle progress (1.0 = failure)', ylabel='Standardized mean value')
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'fd001_normalized_degradation_trends.png', dpi=160, bbox_inches='tight')
plt.show()

## Healthy versus near-failure measurements

For this exploratory comparison, `healthy` means RUL above 120 cycles and `near_failure` means RUL at or below 30 cycles. These are analysis bands, not approved aviation-maintenance limits.

In [ ]:
healthy = engine.loc[engine['rul'] > 120, variable_sensors].mean()
near_failure = engine.loc[engine['rul'] <= 30, variable_sensors].mean()
pooled_std = engine[variable_sensors].std().replace(0, np.nan)
condition_shift = ((near_failure - healthy) / pooled_std).sort_values(key=lambda values: values.abs(), ascending=False)
comparison = pd.DataFrame({
    'healthy_mean': healthy,
    'near_failure_mean': near_failure,
    'standardized_shift': condition_shift,
}).loc[condition_shift.index]
comparison

In [ ]:
summary = pd.Series({
    'subset': SUBSET,
    'rows': len(engine),
    'engines': engine['unit_id'].nunique(),
    'lifetime_min': int(lifetimes['lifetime_cycles'].min()),
    'lifetime_median': float(lifetimes['lifetime_cycles'].median()),
    'lifetime_max': int(lifetimes['lifetime_cycles'].max()),
    'constant_sensor_count': len(constant_sensors),
    'variable_sensor_count': len(variable_sensors),
    'strongest_rul_sensor': rul_correlations.index[0],
    'strongest_rul_correlation': float(rul_correlations.iloc[0]),
}, name='FD001 EDA summary')
summary.to_frame()

## Modeling implications

- Split future validation data by `unit_id`; never randomly split individual cycles.
- Remove constant columns through a training-fitted variance filter.
- Scale variable sensors using training engines only.
- Use lifecycle trends and rolling features carefully, without looking into future cycles.
- Treat the RUL bands used above as exploratory labels only.
- Keep anonymized sensor names in the dashboard because the source does not supply physical identities or units.